# Explainable AI (XAI) Analysis with LIME and Attention


This notebook applies XAI techniques to explain the decisions of fine-tuned models on all processed datasets. The approaches used are LIME (Local Interpretable Model-agnostic Explanations) and Transformer attention visualization.

## Importing Libraries and Utilities

In [1]:
# Import system libraries and set up path for utility modules
import sys
import os

# Add the "src" directory to sys.path to import pipeline utility functions
sys.path.append(os.path.abspath(os.path.join(os.pardir, "src")))

In [2]:
# Import required libraries
import gc
import os
import random
import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import (
    BertForSequenceClassification,
    BertTokenizer,
    RobertaForSequenceClassification,
    RobertaTokenizer,
)

In [3]:
# Import pipeline utility functions for XAI
from xai import (
    get_attention_weights,
    lime_explain_instance,
    plot_lime_explanation,
    plot_mean_attention_heatmap,
    get_lime_top_words,
    get_top_attention_tokens,
)

## General Definitions

In [4]:
# Models and datasets
MODELS = [
    ("google-bert/bert-base-uncased", BertForSequenceClassification, BertTokenizer),
    ("mental/mental-roberta-base", RobertaForSequenceClassification, RobertaTokenizer),
]

DATASETS = [
    "suicide-and-depression-detection",
    "sentiment-analysis-for-mental-health",
    "sentimental-analysis-for-tweets",
    "mental-health-corpus",
]

In [5]:
# Hyperparameters
MAX_LENGTH = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Auxiliary Functions for Sample Selection and Execution

In [6]:
def select_samples(df, n_per_class=1, label_col="label", seed=42):
    """
    Select random samples from a DataFrame for each class.

    Args:
        df (pd.DataFrame): DataFrame containing the data.
        n_per_class (int): Number of samples to select per class.
        label_col (str): Name of the column containing class labels.
        seed (int): Seed for reproducibility.

    Returns:
        samples (list): List of dictionaries, each representing a selected sample.
    """
    random.seed(seed)

    samples = []
    for label in df[label_col].unique():
        subset = df[df[label_col] == label]
        if len(subset) > 0:
            samples.extend(subset.sample(n=min(n_per_class, len(subset)), random_state=seed).to_dict("records"))

    return samples

In [7]:
def xai_analysis_for_samples(
    samples,
    class_names,
    base_path,
    models,
    device,
    max_length=256,
    analysis_type="both",
    output_type="both",
    top_n=10,
    layer=-1,
):
    """
    Run Explainable AI (XAI) analysis using LIME, Attention, or both for a list of samples and multiple models.

    Args:
        samples (list): List of dicts, each with 'clean_text' and 'label'.
        class_names (list): List of class names for the problem.
        base_path (str): Base path where model and tokenizer directories are saved.
        models (list): List of tuples (model_name, model_cls, tokenizer_cls) for each model to evaluate.
        device (torch.device): Device (CPU or GPU) for model execution.
        max_length (int, optional): Max tokenization length. Default: 256.
        analysis_type (str, optional): 'lime', 'attention', or 'both'. Default: 'both'.
        output_type (str, optional): 'numeric', 'figure', or 'both'. Default: 'both'.
        top_n (int, optional): Number of most influential words/tokens to display. Default: 10.
        layer (int, optional): Layer index for attention analysis. Default: -1 (last).
    """
    for model_name, model_cls, tokenizer_cls in models:
        print(f"--- Model: {model_name} ---\n")

        # Model and tokenizer directories
        model_dir = f"{base_path}/{model_name.replace('/', '_')}_hf/model"
        tokenizer_dir = f"{base_path}/{model_name.replace('/', '_')}_hf/tokenizer"
        if not os.path.exists(model_dir):
            print(f"Model not found: {model_dir}")
            continue

        # Load model and tokenizer
        model = model_cls.from_pretrained(model_dir, attn_implementation="eager")
        tokenizer = tokenizer_cls.from_pretrained(tokenizer_dir)
        model.to(device)
        model.eval()

        # Model evaluation
        for i, sample in enumerate(samples):
            text = sample["clean_text"]
            label = sample["label"]

            print(f"Sample {i + 1}:\nText: {text}\nTrue class: {label}")

            # LIME
            if analysis_type in ('lime', 'both'):
                print("\n[LIME]")

                try:
                    # Get predicted class by the model
                    with torch.no_grad():
                        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(device)
                        outputs = model(**inputs)
                        logits = (
                            outputs.logits if hasattr(outputs, "logits") else outputs[0]
                        )
                        pred_index = int(torch.argmax(logits, dim=-1).cpu().numpy().item())
                    label_index = pred_index

                    explanation = lime_explain_instance(
                        text=text,
                        model=model,
                        tokenizer=tokenizer,
                        class_names=class_names,
                        max_length=max_length,
                        device=device,
                    )

                    if output_type in ("figure", "both"):
                        plot_lime_explanation(explanation, label_index=label_index)

                    if output_type in ("numeric", "both"):
                        top_words = get_lime_top_words(explanation, label_index=label_index, top_n=top_n)
                        print("Most influential words by LIME (predicted class):")
                        for word, weight in top_words:
                            print(f"  {word}: {weight:.4f}")

                except Exception as e:
                    print(f"Error generating LIME explanation: {e}")

            # Attention
            if analysis_type in ("attention", "both"):
                print("\n[Attention]")

                try:
                    tokens, attn_weights = get_attention_weights(
                        model=model,
                        tokenizer=tokenizer,
                        text=text,
                        max_length=max_length,
                        device=device,
                    )

                    if output_type in ("figure", "both"):
                        plot_mean_attention_heatmap(tokens, attn_weights, layer=layer)

                    if output_type in ("numeric", "both"):
                        top_attention = get_top_attention_tokens(tokens, attn_weights, layer=layer, top_n=top_n)
                        print("Most attended tokens (received attention):")
                        for token, attn in top_attention:
                            print(f"  {token}: {attn:.4f}")

                except Exception as e:
                    print(f"Error generating attention visualization: {e}")

            # Free memory, GPU, and close figures
            del text, label
            if "explanation" in locals():
                del explanation
            if "tokens" in locals():
                del tokens
            if "attn_weights" in locals():
                del attn_weights
            torch.cuda.empty_cache()
            gc.collect()
            plt.close("all")

            print("\n" + "-"*60 + "\n")

## XAI - Numeric Visualizations (Values)

### Suicide and Depression Detection Dataset

In [8]:
# Load test data for Suicide and Depression Detection
DATA_DIR = "../data/processed/suicide-and-depression-detection"
TEST_DF = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

In [9]:
# Select samples for XAI analysis
class_names = sorted(TEST_DF["label"].unique())
samples = select_samples(TEST_DF, n_per_class=2, label_col="label")

In [10]:
# Base directory for models
BASE_PATH = "../models/suicide-and-depression-detection"

In [11]:
# Run LIME XAI analysis (numeric output)
xai_analysis_for_samples(
    samples=samples,
    class_names=class_names,
    base_path=BASE_PATH,
    models=MODELS,
    device=DEVICE,
    max_length=MAX_LENGTH,
    analysis_type="lime",
    output_type="numeric",
    top_n=20,
    layer=0
)

--- Model: google-bert/bert-base-uncased ---

Sample 1:
Text: i dont see a purpose anymorei’ve tried coming up with things i’m excited about the future, things i want to live through. but i really cant come up with anything and im at the point where nothing matters to me anymore and i’m too fucking tired to do anything about anything. i had someone i could talk to about these thoughts but a week after i opened up to them they stopped talking to me for no apparent reason. thats probably another reason i want to kill myself. no one cares about me as much as they say they do. i’ve been spiraling for a while now and no one has noticed or cared to notice. i just dont see a purpose in living anymore but im too scared of failing when i attempt to do it.
True class: suicide

[LIME]
Most influential words by LIME (predicted class):
  anymorei: 0.0308
  been: 0.0166
  tried: 0.0164
  attempt: 0.0155
  kill: 0.0140
  live: 0.0138
  for: 0.0126
  about: -0.0119
  i: -0.0113
  excited: -0.0065

---

In [12]:
# Run Attention XAI analysis (numeric output)
xai_analysis_for_samples(
    samples=samples,
    class_names=class_names,
    base_path=BASE_PATH,
    models=MODELS,
    device=DEVICE,
    max_length=MAX_LENGTH,
    analysis_type="attention",
    output_type="numeric",
    top_n=20,
    layer=0,
)

--- Model: google-bert/bert-base-uncased ---

Sample 1:
Text: i dont see a purpose anymorei’ve tried coming up with things i’m excited about the future, things i want to live through. but i really cant come up with anything and im at the point where nothing matters to me anymore and i’m too fucking tired to do anything about anything. i had someone i could talk to about these thoughts but a week after i opened up to them they stopped talking to me for no apparent reason. thats probably another reason i want to kill myself. no one cares about me as much as they say they do. i’ve been spiraling for a while now and no one has noticed or cared to notice. i just dont see a purpose in living anymore but im too scared of failing when i attempt to do it.
True class: suicide

[Attention]
Most attended tokens (received attention):
  [CLS]: 6.6252
  scared: 1.9590
  ’: 1.8380
  anymore: 1.6961
  ’: 1.6938
  matters: 1.6181
  ’: 1.6143
  cared: 1.5955
  ’: 1.5933
  excited: 1.5857
  fucking: 1.577

### Sentiment Analysis for Mental Health Dataset

In [13]:
# Load test data for Sentiment Analysis for Mental Health
DATA_DIR = "../data/processed/sentiment-analysis-for-mental-health"
TEST_DF = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

In [14]:
# Select samples for XAI analysis
class_names = sorted(TEST_DF["label"].unique())
samples = select_samples(TEST_DF, n_per_class=2, label_col="label")

In [15]:
# Base directory for models
BASE_PATH = "../models/sentiment-analysis-for-mental-health"

In [16]:
# Run LIME XAI analysis (numeric output)
xai_analysis_for_samples(
    samples=samples,
    class_names=class_names,
    base_path=BASE_PATH,
    models=MODELS,
    device=DEVICE,
    max_length=MAX_LENGTH,
    analysis_type="lime",
    output_type="numeric",
    top_n=20,
    layer=0,
)

--- Model: google-bert/bert-base-uncased ---

Sample 1:
Text: alystoe i hope you are okay
True class: normal

[LIME]
Most influential words by LIME (predicted class):
  alystoe: -0.0004
  hope: -0.0003
  are: -0.0001
  i: 0.0001
  okay: -0.0001
  you: 0.0000

------------------------------------------------------------

Sample 2:
Text: jennchambless me neither and nobody is awake nobody i m drunk and alone
True class: normal

[LIME]
Most influential words by LIME (predicted class):
  jennchambless: -0.0006
  awake: -0.0004
  nobody: 0.0004
  m: -0.0004
  alone: 0.0003
  neither: -0.0002
  and: 0.0001
  drunk: 0.0001
  is: -0.0001
  me: 0.0000

------------------------------------------------------------

Sample 3:
Text: that is all i fucking need i just want somebody to talk to me please somebody please talk to me
True class: suicide

[LIME]
Most influential words by LIME (predicted class):
  somebody: -0.0001
  talk: -0.0001
  please: -0.0001
  is: -0.0001
  that: -0.0001
  just: -0.0

In [17]:
# Run Attention XAI analysis (numeric output)
xai_analysis_for_samples(
    samples=samples,
    class_names=class_names,
    base_path=BASE_PATH,
    models=MODELS,
    device=DEVICE,
    max_length=MAX_LENGTH,
    analysis_type="attention",
    output_type="numeric",
    top_n=20,
    layer=0,
)

--- Model: google-bert/bert-base-uncased ---

Sample 1:
Text: alystoe i hope you are okay
True class: normal

[Attention]
Most attended tokens (received attention):
  [CLS]: 2.2812
  okay: 1.0953
  [SEP]: 1.0470
  ##yst: 0.9825
  hope: 0.9731
  ##oe: 0.7663
  i: 0.7576
  you: 0.7464
  al: 0.7085
  are: 0.6421

------------------------------------------------------------

Sample 2:
Text: jennchambless me neither and nobody is awake nobody i m drunk and alone
True class: normal

[Attention]
Most attended tokens (received attention):
  [CLS]: 3.0139
  [SEP]: 1.1639
  jen: 1.1476
  ##bles: 1.1368
  nobody: 1.1265
  nobody: 1.1056
  drunk: 1.0921
  awake: 1.0844
  neither: 0.9876
  me: 0.7873
  alone: 0.7783
  i: 0.7614
  ##nch: 0.7602
  m: 0.7136
  ##am: 0.7088
  and: 0.6843
  is: 0.6799
  ##s: 0.6610
  and: 0.6068

------------------------------------------------------------

Sample 3:
Text: that is all i fucking need i just want somebody to talk to me please somebody please talk to me
Tr

### Sentimental Analysis for Tweets Dataset

In [18]:
# Load test data for Sentimental Analysis for Tweets
DATA_DIR = "../data/processed/sentimental-analysis-for-tweets"
TEST_DF = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

In [19]:
# Select samples for XAI analysis
class_names = sorted(TEST_DF["label"].unique())
samples = select_samples(TEST_DF, n_per_class=2, label_col="label")

In [20]:
# Base directory for models
BASE_PATH = "../models/sentimental-analysis-for-tweets"

In [21]:
# Run LIME XAI analysis (numeric output)
xai_analysis_for_samples(
    samples=samples,
    class_names=class_names,
    base_path=BASE_PATH,
    models=MODELS,
    device=DEVICE,
    max_length=MAX_LENGTH,
    analysis_type="lime",
    output_type="numeric",
    top_n=20,
    layer=0,
)

--- Model: google-bert/bert-base-uncased ---

Sample 1:
Text: they make me feel bad for having depression like????
True class: depression

[LIME]
Most influential words by LIME (predicted class):
  depression: -0.9759
  feel: -0.0075
  me: 0.0056
  they: -0.0047
  bad: -0.0026
  make: 0.0014
  for: 0.0013
  like: 0.0009
  having: -0.0005

------------------------------------------------------------

Sample 2:
Text: i certainly do enjoy havimg depression :)
True class: depression

[LIME]
Most influential words by LIME (predicted class):
  depression: -0.9865
  enjoy: 0.0013
  certainly: 0.0013
  i: 0.0009
  do: 0.0006
  havimg: 0.0002

------------------------------------------------------------

Sample 3:
Text: life &amp; style is dumb. &quot;twilight heartthrob robert pattinson cozies up to a blonde in cannes - what will his crush kristen stewart think?&quot;
True class: not_depression

[LIME]
Most influential words by LIME (predicted class):
  his: 0.0001
  life: -0.0001
  blonde: -0

In [22]:
# Run Attention XAI analysis (numeric output)
xai_analysis_for_samples(
    samples=samples,
    class_names=class_names,
    base_path=BASE_PATH,
    models=MODELS,
    device=DEVICE,
    max_length=MAX_LENGTH,
    analysis_type="attention",
    output_type="numeric",
    top_n=20,
    layer=0,
)

--- Model: google-bert/bert-base-uncased ---

Sample 1:
Text: they make me feel bad for having depression like????
True class: depression

[Attention]
Most attended tokens (received attention):
  [CLS]: 1.9197
  feel: 1.3649
  depression: 1.1413
  [SEP]: 1.0573
  ?: 1.0215
  ?: 0.9806
  ?: 0.9557
  bad: 0.9065
  ?: 0.9020
  make: 0.8855
  having: 0.8726
  like: 0.7967
  me: 0.7848
  for: 0.7169
  they: 0.6938

------------------------------------------------------------

Sample 2:
Text: i certainly do enjoy havimg depression :)
True class: depression

[Attention]
Most attended tokens (received attention):
  [CLS]: 2.4154
  [SEP]: 1.1589
  enjoy: 1.1161
  certainly: 1.0386
  depression: 0.9894
  :: 0.8637
  i: 0.8352
  ha: 0.8208
  ##mg: 0.7503
  ): 0.7133
  ##vi: 0.6732
  do: 0.6250

------------------------------------------------------------

Sample 3:
Text: life &amp; style is dumb. &quot;twilight heartthrob robert pattinson cozies up to a blonde in cannes - what will his crush kris

### Mental Health Corpus Dataset

In [23]:
# Load test data for Mental Health Corpus
DATA_DIR = "../data/processed/mental-health-corpus"
TEST_DF = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

In [24]:
# Select samples for XAI analysis
class_names = sorted(TEST_DF["label"].unique())
samples = select_samples(TEST_DF, n_per_class=2, label_col="label")

In [25]:
# Base directory for models
BASE_PATH = "../models/mental-health-corpus"

In [26]:
# Run LIME XAI analysis (numeric output)
xai_analysis_for_samples(
    samples=samples,
    class_names=class_names,
    base_path=BASE_PATH,
    models=MODELS,
    device=DEVICE,
    max_length=MAX_LENGTH,
    analysis_type="lime",
    output_type="numeric",
    top_n=20,
    layer=0,
)

--- Model: google-bert/bert-base-uncased ---

Sample 1:
Text: seen several comments brando using southern accent felt mistake movie made racism discrimination strong south jim crow laws still effect civil rights infancy could possibly subtle social commentary southern man love woman another race way mash subtle criticism viet nam war thoughtsbr br another comment made myoshi umeki appearing cold anyone japan would understand japanese people least experience tend show emotion front strangers strict social rules especially men meeting single women americans japan totally foreign culture blunt attempts meet women shocking ladies one trait japanese smile embarrassed uncomfortable many american servicemen took sign advances welcomed also remember time represented movie japan defeated occupying forces treated reluctant acceptance think myoshi umeki gave credible performance situation would been watching interaction american actors brought back several memories experiences country able meet p

In [27]:
# Run Attention XAI analysis (numeric output)
xai_analysis_for_samples(
    samples=samples,
    class_names=class_names,
    base_path=BASE_PATH,
    models=MODELS,
    device=DEVICE,
    max_length=MAX_LENGTH,
    analysis_type="attention",
    output_type="numeric",
    top_n=20,
    layer=0,
)

--- Model: google-bert/bert-base-uncased ---

Sample 1:
Text: seen several comments brando using southern accent felt mistake movie made racism discrimination strong south jim crow laws still effect civil rights infancy could possibly subtle social commentary southern man love woman another race way mash subtle criticism viet nam war thoughtsbr br another comment made myoshi umeki appearing cold anyone japan would understand japanese people least experience tend show emotion front strangers strict social rules especially men meeting single women americans japan totally foreign culture blunt attempts meet women shocking ladies one trait japanese smile embarrassed uncomfortable many american servicemen took sign advances welcomed also remember time represented movie japan defeated occupying forces treated reluctant acceptance think myoshi umeki gave credible performance situation would been watching interaction american actors brought back several memories experiences country able meet p

## Final Remarks


- The adopted strategies highlighted keywords associated with the model's decisions, with consistent patterns across models and datasets. After fine-tuning, the models tend to prioritize salient information early in the processing.
- The attention analysis suggests that earlier layers are generally more interpretable, with concentrated attention on semantically relevant tokens. In deeper layers, attention becomes more diffuse, making interpretation less straightforward.
- XAI techniques provide useful indications of model focus on relevant keywords and nuances, but they do not fully explain the model’s internal reasoning, reinforcing the trade-off between interpretability and performance in deep learning models.